# Samaritan — GRPO, standalone

No `trl`, no `unsloth`, no `mergekit`. Only **torch, transformers, peft** —
which already work in this runtime; it was always the framework wrapper that
died, never the model.

GRPO written out is two pages: sample G completions per prompt, grade them, and
set each one's advantage to how far its reward sits from its *group's* mean in
units of the group's spread. The siblings are the baseline. No critic, no value
head, no reference model.

**Runtime → Change runtime type → A100.** An L4 will be tight: bf16 weights are
~8 GB and the generation KV cache for 8 concurrent completions adds ~10 GB.

### What to expect

Rollouts are plain HF `generate` — no vLLM — so a step is **minutes, not
seconds**: 8 completions of up to 8k tokens is ~65k tokens each step. The
default 40 steps is a run that shows whether reward moves. It is not a finished
model, and it is not meant to be.

### Why it trains the BASE

Measured 2026-09-16 at a budget where both models finish: base **39/40**, the
185-trace SFT student **30/40**, McNemar p = 0.012. The student is not more
concise — it fails to *terminate*, running past 16,000 tokens on nine of forty
items where the base never exceeds 7,121. A verifiable reward scores a runaway
rollout 0, so GRPO attacks that defect directly.

## 1. Drive, and the bundle

Drive is mounted first because **checkpoints go there, not to `/content`**.
Colab preempts, and anything under `/content` dies with the runtime — an
unattended run that gets reclaimed at step 30 would otherwise leave nothing at
all. With Drive, re-running the training cell picks up where it stopped.

In [ ]:
import os, glob, subprocess, sys, time

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/samaritan-grpo'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

if not glob.glob('grpo-bundle.tar.gz'):
    from google.colab import files
    print('Select grpo-bundle.tar.gz (in your models folder)...')
    files.upload()
assert os.path.exists('grpo-bundle.tar.gz'), 'bundle not uploaded'
!rm -rf samaritan && mkdir -p samaritan && tar -xzf grpo-bundle.tar.gz -C samaritan
print(sorted(os.listdir('samaritan/training')))

## 2. The maths, verified before anything else

Runs on CPU in a second. If the advantage sign or the length normalisation were
wrong, the model would train toward exactly what the grader rejects while the
loss curve looked perfectly healthy — so this runs first, every time.

In [ ]:
!cd samaritan && python training/test_grpo_standalone.py

## 3. The reward — the harness's own Rust grader

Not a Python reimplementation. A second oracle that silently disagrees with the
grader every measurement in this project used is how a model gets optimised
toward the wrong target without anyone noticing.

In [ ]:
t0 = time.time()
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q',
                   shell=True, check=True)
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

!cd samaritan && cargo build --release -q -p samaritan-corpus --example grade_batch
!cd samaritan && cargo build --release -q -p samaritan-curriculum --example generate

def built(name):
    base = f'samaritan/target/release/examples/{name}'
    hits = [base] if os.path.exists(base) else sorted(glob.glob(base + '-*'))
    hits = [h for h in hits if not h.endswith('.d')]
    assert hits, f'{name} did not build'
    return os.path.abspath(hits[0])

GRADER, GENERATE = built('grade_batch'), built('generate')
print(f'built in {time.time()-t0:.0f}s')

import json
probe = '\n'.join([
    json.dumps({'given': 'Answer: 42', 'answer': '42', 'answer_kind': 'exactMatch'}),
    json.dumps({'given': 'Answer: 43', 'answer': '42', 'answer_kind': 'exactMatch'}),
])
out = subprocess.run([GRADER], input=probe, capture_output=True, text=True)
v = [json.loads(l)['correct'] for l in out.stdout.splitlines() if l.strip()]
assert v == [True, False], f'grader is wrong: {v} / {out.stderr[:300]}'
print('grader verified:', v)

## 4. Training problems

Seed 1001 — not 777/883/884, which are eval seeds. Same generator, disjoint
problems, so training cannot touch the measurement set by construction rather
than by trusting a filter.

In [ ]:
os.environ.update(SEED='1001', DIFFICULTY='3', COUNT='400',
                  SPLIT_LABEL='train', OUT='/content/grpo-train-d3.jsonl')
!{GENERATE}

import collections
rows = [json.loads(l) for l in open('/content/grpo-train-d3.jsonl') if l.strip()]
fam = collections.Counter(r['id'].split('-')[1] for r in rows)
print(f'\n{len(rows)} problems across {len(fam)} families: {dict(fam)}')

## 5. Dependencies, checked the way training uses them

`peft` is the only install; torch and transformers are already here. The check
runs in a **subprocess**, importing exactly what the trainer imports — an import
that works in this kernel proves nothing about the separate interpreter that
runs the script, which is how four earlier runs reached a model load before
dying on a missing package.

In [ ]:
!pip -q install peft 2>&1 | tail -2

PROBE = "import torch, transformers, peft\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom peft import LoraConfig, get_peft_model\nprint('  torch', torch.__version__, '| transformers', transformers.__version__,\n      '| peft', peft.__version__)\nok = torch.cuda.is_available()\nprint('  cuda', ok, '|', torch.cuda.get_device_name(0) if ok else 'NO GPU')\nif ok:\n    free, total = torch.cuda.mem_get_info()\n    print(f'  vram {free/1e9:.1f} GB free of {total/1e9:.1f} GB')\nassert ok, 'no GPU - Runtime > Change runtime type'"
r = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
print(r.stdout, end='')
if r.returncode != 0:
    raise SystemExit('training imports FAILED in a subprocess:\n' + r.stderr[-2500:])
print('\nall training imports resolve in a subprocess')

## 6. Dry run — dataset and reward path, no GPU

Proves the trainer can read the problems and reach the grader, before a model is
loaded. A reward function that silently returns 0 trains the model to do nothing,
slowly and expensively.

In [ ]:
!cd samaritan && python training/grpo_standalone.py /content/grpo-train-d3.jsonl \
    --grader {GRADER} --dry-run

## 7. Smoke test — 2 steps

The first run that touches the GPU. Two steps prove generation, grading and a
backward pass all work together in *this* runtime. Roughly five minutes; if it
fails, it fails here rather than an hour into the real run.

Small on purpose — 2 completions of 512 tokens — so it exercises every code path
cheaply. Expect reward 0.00 or 1.00 and a skipped step: a flat group is normal at
this size, and the trainer skipping it is correct behaviour, not a failure.

In [ ]:
!cd samaritan && python training/grpo_standalone.py /content/grpo-train-d3.jsonl \
    --grader {GRADER} --steps 2 --generations 2 --max-new 512 \
    --save-every 0 --output /content/adapters/smoke

## 8. Train

**Watch the `reward` column, not the loss.** If mean reward climbs over the first
~15 steps, RL is working.

Around step 10 a reward-health line prints. If it reports no gradient it also says
*which* cause, because the two need opposite fixes: rollouts with no `Answer:`
line are being cut off (raise `--max-new`), while rollouts that finish and still
agree mean the problems are uniformly too easy or too hard (change `DIFFICULTY`).
Neither is fixed by training longer.

Checkpoints land in Drive every 5 steps and the log is tee'd there too, so
both survive the runtime. If Colab reclaims the session, re-run this one cell:
`--resume` continues the adapter rather than starting over.

Two things now stop a wasted night rather than reporting one:

- **a flat reward aborts** at step 10 instead of spending the rest of the run on
  a zero gradient, and prints which of the two causes it was
- **CUDA OOM halves the group and retries** rather than killing the run — a
  smaller group still teaches something; a crash at step 3 teaches nothing

In [ ]:
# --resume: re-running this cell after a preemption continues the adapter
#   already in Drive rather than starting over from random weights.
# 2>&1 | tee: the log lands in Drive too, so the outcome is readable from a
#   phone even if the runtime is long gone.
!cd samaritan && python training/grpo_standalone.py /content/grpo-train-d3.jsonl \
    --grader {GRADER} --steps 40 --generations 8 --max-new 8192 \
    --save-every 5 --resume \
    --output {OUT}/reasoning-grpo 2>&1 | tee -a {OUT}/train.log

## 9. Result

Everything is already in Drive - nothing here needs you at the keyboard. This
just prints what landed, so the outcome is legible when you come back.

In [ ]:
adapter = f'{OUT}/reasoning-grpo'
print('adapter:', adapter)
for f in sorted(glob.glob(adapter + '/*')):
    print(f'  {os.path.basename(f):<28} {os.path.getsize(f)/1e6:>8.2f} MB')

# The reward trace is the result. Flat means it learned nothing, whatever the
# loss did.
log = f'{OUT}/train.log'
if os.path.exists(log):
    steps = [l for l in open(log, encoding='utf-8', errors='replace')
             if l.startswith('step')]
    print(f'\n{len(steps)} steps logged. First and last five:')
    for l in steps[:5] + (['  ...'] if len(steps) > 10 else []) + steps[-5:]:
        print('  ' + l.rstrip())

print('\nDownload it whenever you are back:')
print('  from google.colab import files')
print(f'  files.download(\'{OUT}/train.log\')')

## Then

It is a LoRA adapter, not a GGUF — merge it into the base and convert before
serving through Ollama, then A/B it against the base with the settings that
finally produced a clean measurement:

```
-Limit 40 -Tag "-grpo" -Shards 3      # MaxTokens/Ctx/Seed are correct by default
```

The bar is the base's **39/40**. Anything that does not clear it is not progress,
however good the reward curve looked.